In [1]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split


In [2]:
data = pd.read_csv('../data/advertising.csv')

train_data, test_data = train_test_split(data, test_size=0.2)

X_train = train_data[['TV', 'Radio', 'Newspaper']]
y_train = train_data['Sales']
X_test = test_data[['TV', 'Radio', 'Newspaper']]
y_test = test_data['Sales']

In [3]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)

In [4]:
for i, col in enumerate(X_train_scaled.columns):
    plt.figure(figsize=(10, 6))
    plt.scatter(X_train_scaled[col], y_train)
    plt.title(f"Scatter plot of {col} vs Target")
    plt.xlabel(col)
    plt.ylabel("Target")
    plt.tight_layout()
    # plt.show()
    plt.close()

In [5]:
train_data_for_plot = X_train.copy()
train_data_for_plot['Sales'] = y_train

sns.pairplot(train_data_for_plot)
# plt.show()
plt.close()

Jest zauważalna liniowa korelacja TV i Sales, tutaj kombinowanie z regresją może mieć sens

In [6]:
from sklearn.linear_model import LinearRegression
import sklearn.metrics as skm

model_lr_raw_data = LinearRegression()
model_lr_scaled_data = LinearRegression()

In [7]:
def model_train_eval(model, X_, y_, X_test_, y_test_, variant):
    model.fit(X_, y_)

    print(f"Coefficients: {model.coef_}, Intercept: {model.intercept_}")

    y_pred = model.predict(X_test_)
    rsc = model.score(X_test_, y_test_)
    mse = skm.mean_squared_error(y_test_, y_pred)
    mape = skm.mean_absolute_percentage_error(y_test, y_pred)
    mae = skm.mean_absolute_error(y_test, y_pred)

    return dict(variant=variant, MSE=mse, MAE=mae, MAPE=mape, Rsq=rsc, Coefficients=model.coef_, Intercept=model.intercept_)

In [8]:
all_results = []

all_results.append(model_train_eval(LinearRegression(fit_intercept=True), X_train, y_train, X_test, y_test, "raw"))
all_results.append(model_train_eval(LinearRegression(fit_intercept=True), X_train_scaled, y_train, X_test_scaled, y_test, "scaled"))

results = pd.DataFrame(all_results)
results

Coefficients: [ 0.05480338  0.1069984  -0.00110472], Intercept: 4.624592227317979
Coefficients: [ 4.78836545  1.63261093 -0.02506211], Intercept: 15.119375000000002


,variant,MSE,MAE,MAPE,Rsq,Coefficients,Intercept
0,raw,1.738271,1.020312,0.082144,0.92216,"[0.054803378706289346, 0.10699839588278187, -0...",4.624592
1,scaled,1.738271,1.020312,0.082144,0.92216,"[4.788365452806496, 1.632610926616744, -0.0250...",15.119375


Wyniki te same ...